# Chapter 5 Practical: Hybrid, Production, and Emerging Recommender Systems

This notebook follows `RS_C5_New_V1.pdf`: Hybrid, Production, and Emerging Recommender Systems.

Learning objectives:
- Understand why real systems combine several recommendation signals.
- Implement simple weighted, switching, mixed, and cascade hybrids.
- Build a small two-stage pipeline: candidates → ranking → re-ranking.
- Add a short explanation for a recommendation.
- Connect classroom prototypes to production and responsible deployment ideas.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_05_hybrid_production_emerging/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_05_hybrid_production_emerging/data"


def read_chapter5_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)


ratings = read_chapter5_csv("ratings_chapter5.csv")
movies = read_chapter5_csv("movies_chapter5.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix = rating_matrix.reindex(columns=movies["title"].tolist())
rating_matrix


## Packages and key functions used

- `pandas` is used for tables: `read_csv`, `merge`, `pivot_table`, `groupby`, and `sort_values`.
- `numpy` is used for score combination and normalization.
- `sklearn.metrics.pairwise.cosine_similarity` is used for a simple content-based score.
- Helper functions below build CF scores, CB scores, and hybrid lists that match the lecture strategies.


## Part 1 - Why no single approach wins

From the lecture:

- Content-based RS understands item information and helps with new items.
- Collaborative filtering learns from user behaviour and can find unexpected preferences.
- Real systems combine several signals for accuracy, coverage, diversity, and robustness.

In this notebook we use a small movie platform scenario:

- many users have sparse ratings,
- Nora is almost a new user,
- `City Laughs` is a new movie with no ratings yet,
- movie descriptions and genres support content-based scoring.


In [ ]:
n_users, n_items = rating_matrix.shape
n_known = rating_matrix.notna().sum().sum()
density = n_known / (n_users * n_items)

user_counts = rating_matrix.notna().sum(axis=1).rename("n_ratings")
item_counts = rating_matrix.notna().sum(axis=0).rename("n_ratings")

summary = pd.DataFrame(
    {
        "measure": ["users", "items", "known ratings", "possible ratings", "density"],
        "value": [n_users, n_items, n_known, n_users * n_items, round(density, 3)],
    }
)
display(summary)
display(user_counts.to_frame())
display(item_counts.to_frame().T)
movies[["title", "genre", "year", "popularity"]]


## Part 2 - Build simple CF and CB scores

We create two separate recommendation signals:

1. **Collaborative Filtering (CF):** item-item cosine similarity on the rating matrix.
2. **Content-Based (CB):** cosine similarity over genre one-hot features and a small description bag-of-words.

Scores are later normalized so a weighted hybrid can combine them fairly.


In [ ]:
def normalize_scores(series):
    series = series.astype(float)
    if series.empty:
        return series
    min_v, max_v = series.min(), series.max()
    if max_v == min_v:
        return series * 0 + 0.5
    return (series - min_v) / (max_v - min_v)


def cf_scores_for_user(user_id, rating_matrix):
    filled = rating_matrix.fillna(0.0)
    item_sim = pd.DataFrame(
        cosine_similarity(filled.T),
        index=filled.columns,
        columns=filled.columns,
    )
    user_ratings = rating_matrix.loc[user_id].dropna()
    if user_ratings.empty:
        return pd.Series(dtype=float)

    scores = {}
    for item in rating_matrix.columns:
        if item in user_ratings.index:
            continue
        numer = 0.0
        denom = 0.0
        for seen_item, rating in user_ratings.items():
            sim = item_sim.loc[item, seen_item]
            if sim > 0:
                numer += sim * rating
                denom += sim
        if denom > 0:
            scores[item] = numer / denom
    return pd.Series(scores, dtype=float).sort_values(ascending=False)


def build_content_features(movies_df):
    genre_dummies = pd.get_dummies(movies_df["genre"])
    tokens = movies_df["description"].str.lower().str.replace(r"[^a-z\s]", " ", regex=True)
    vocab = sorted({w for text in tokens for w in text.split() if len(w) > 3})
    bow = pd.DataFrame(
        [{w: text.split().count(w) for w in vocab} for text in tokens],
        index=movies_df["title"],
    )
    features = pd.concat([genre_dummies.set_index(movies_df["title"]), bow], axis=1)
    return features.astype(float)


content_features = build_content_features(movies)


def cb_scores_for_user(user_id, rating_matrix, content_features, liked_threshold=4.0):
    user_ratings = rating_matrix.loc[user_id].dropna()
    liked = user_ratings[user_ratings >= liked_threshold]
    if liked.empty:
        liked = user_ratings
    if liked.empty:
        return pd.Series(dtype=float)

    profile = content_features.loc[liked.index].mean()
    candidates = [t for t in content_features.index if t not in user_ratings.index]
    if not candidates:
        return pd.Series(dtype=float)

    sims = cosine_similarity(
        profile.values.reshape(1, -1),
        content_features.loc[candidates].values,
    ).flatten()
    return pd.Series(sims, index=candidates).sort_values(ascending=False)


target_user = "Anna"
cf_anna = cf_scores_for_user(target_user, rating_matrix)
cb_anna = cb_scores_for_user(target_user, rating_matrix, content_features)

pd.DataFrame(
    {
        "cf_score": cf_anna,
        "cb_score": cb_anna,
        "cf_norm": normalize_scores(cf_anna),
        "cb_norm": normalize_scores(cb_anna),
    }
).fillna("-")


## Part 3 - Weighted hybrid

Lecture formula idea:

$$
FinalScore = w_{CF} \cdot CF + w_{CB} \cdot CB
$$

Example from the slides: `0.6 CF + 0.4 CB`.

Important classroom rule: normalize CF and CB scores before combining them.


In [ ]:
def weighted_hybrid(user_id, rating_matrix, content_features, w_cf=0.6, w_cb=0.4):
    cf = normalize_scores(cf_scores_for_user(user_id, rating_matrix))
    cb = normalize_scores(cb_scores_for_user(user_id, rating_matrix, content_features))
    all_items = sorted(set(cf.index) | set(cb.index))
    rows = []
    for item in all_items:
        cf_v = float(cf.get(item, 0.0))
        cb_v = float(cb.get(item, 0.0))
        rows.append(
            {
                "title": item,
                "cf_norm": cf_v,
                "cb_norm": cb_v,
                "final_score": w_cf * cf_v + w_cb * cb_v,
            }
        )
    return pd.DataFrame(rows).sort_values("final_score", ascending=False).reset_index(drop=True)


weighted_anna = weighted_hybrid("Anna", rating_matrix, content_features, w_cf=0.6, w_cb=0.4)
weighted_anna


## Part 4 - Switching hybrid

A switching hybrid chooses one model for each situation:

- new / cold-start user → popularity or content-based,
- some history → memory-based CF or weighted hybrid,
- rich history → model-based / stronger personalized CF.

Here we use a simple rule on the number of ratings.


In [ ]:
def popularity_scores(movies_df, exclude_titles=None):
    if exclude_titles is None:
        exclude_titles = []
    exclude_titles = set(exclude_titles)
    scores = movies_df.set_index("title")["popularity"].astype(float)
    return scores.drop(labels=list(exclude_titles), errors="ignore").sort_values(ascending=False)


def switching_hybrid(user_id, rating_matrix, content_features, movies_df, cold_threshold=2):
    n_ratings = int(rating_matrix.loc[user_id].notna().sum()) if user_id in rating_matrix.index else 0
    seen = rating_matrix.loc[user_id].dropna().index if user_id in rating_matrix.index else []

    if n_ratings < cold_threshold:
        strategy = "popularity_fallback"
        ranking = popularity_scores(movies_df, exclude_titles=seen).rename("score").reset_index()
        ranking.columns = ["title", "score"]
    elif n_ratings < 4:
        strategy = "content_based"
        ranking = cb_scores_for_user(user_id, rating_matrix, content_features).rename("score").reset_index()
        ranking.columns = ["title", "score"]
    else:
        strategy = "weighted_cf_cb"
        ranking = weighted_hybrid(user_id, rating_matrix, content_features)[["title", "final_score"]]
        ranking = ranking.rename(columns={"final_score": "score"})

    ranking["strategy"] = strategy
    ranking["n_ratings"] = n_ratings
    return ranking.sort_values("score", ascending=False).reset_index(drop=True)


for user in ["Nora", "Anna", "Ben"]:
    print(f"\n=== Switching hybrid for {user} ===")
    display(switching_hybrid(user, rating_matrix, content_features, movies).head(5))


## Part 5 - Mixed hybrid

A mixed hybrid shows results from several methods in one list. Scores are not always combined into one number.

Example Top-8 list inspired by the lecture:

- 3 collaborative filtering items,
- 3 content-based items,
- 1 trending / popular item,
- 1 newly released item.


In [ ]:
def mixed_hybrid(user_id, rating_matrix, content_features, movies_df, n_cf=3, n_cb=3, n_pop=1, n_new=1):
    seen = set(rating_matrix.loc[user_id].dropna().index)
    cf = cf_scores_for_user(user_id, rating_matrix)
    cb = cb_scores_for_user(user_id, rating_matrix, content_features)
    pop = popularity_scores(movies_df, exclude_titles=seen)
    newest = movies_df.set_index("title")["year"].drop(labels=list(seen), errors="ignore").sort_values(ascending=False)

    selected = []
    used = set()

    def take_from(series, source, n):
        count = 0
        for title, score in series.items():
            if title in used:
                continue
            selected.append({"title": title, "source": source, "source_score": float(score)})
            used.add(title)
            count += 1
            if count >= n:
                break

    take_from(cf, "collaborative_filtering", n_cf)
    take_from(cb, "content_based", n_cb)
    take_from(pop, "trending_popular", n_pop)
    take_from(newest, "newly_released", n_new)
    return pd.DataFrame(selected)


mixed_anna = mixed_hybrid("Anna", rating_matrix, content_features, movies)
mixed_anna


## Part 6 - Cascade hybrid and a tiny production pipeline

Production systems often use stages:

1. **Candidate generation** - high recall, reduce a large catalogue quickly.
2. **Ranking** - order candidates more carefully.
3. **Re-ranking** - apply diversity, freshness, or business rules.

This is also the cascade idea: one model searches broadly, the next decides carefully.


In [ ]:
def candidate_generation(user_id, rating_matrix, content_features, movies_df, n_candidates=5):
    cf = cf_scores_for_user(user_id, rating_matrix).head(n_candidates)
    cb = cb_scores_for_user(user_id, rating_matrix, content_features).head(n_candidates)
    pop = popularity_scores(movies_df, exclude_titles=rating_matrix.loc[user_id].dropna().index).head(n_candidates)
    candidates = sorted(set(cf.index) | set(cb.index) | set(pop.index))
    return candidates, {"cf": cf, "cb": cb, "popular": pop}


def rank_candidates(user_id, candidates, rating_matrix, content_features, movies_df, w_cf=0.5, w_cb=0.3, w_pop=0.2):
    cf = normalize_scores(cf_scores_for_user(user_id, rating_matrix))
    cb = normalize_scores(cb_scores_for_user(user_id, rating_matrix, content_features))
    pop = normalize_scores(popularity_scores(movies_df))
    rows = []
    for title in candidates:
        cf_v = float(cf.get(title, 0.0))
        cb_v = float(cb.get(title, 0.0))
        pop_v = float(pop.get(title, 0.0))
        rows.append(
            {
                "title": title,
                "cf_norm": cf_v,
                "cb_norm": cb_v,
                "pop_norm": pop_v,
                "rank_score": w_cf * cf_v + w_cb * cb_v + w_pop * pop_v,
            }
        )
    return pd.DataFrame(rows).sort_values("rank_score", ascending=False).reset_index(drop=True)


def rerank_for_diversity_and_freshness(ranked_df, movies_df, top_k=5):
    meta = movies_df.set_index("title")
    selected = []
    used_genres = set()
    for _, row in ranked_df.iterrows():
        title = row["title"]
        genre = meta.loc[title, "genre"]
        year = int(meta.loc[title, "year"])
        diversity_bonus = 0.15 if genre not in used_genres else 0.0
        freshness_bonus = 0.10 if year >= 2023 else 0.0
        final = row["rank_score"] + diversity_bonus + freshness_bonus
        selected.append(
            {
                "title": title,
                "genre": genre,
                "year": year,
                "rank_score": row["rank_score"],
                "diversity_bonus": diversity_bonus,
                "freshness_bonus": freshness_bonus,
                "final_score": final,
            }
        )
        used_genres.add(genre)
        if len(selected) >= top_k:
            break
    return pd.DataFrame(selected).sort_values("final_score", ascending=False).reset_index(drop=True)


candidates, sources = candidate_generation("Anna", rating_matrix, content_features, movies, n_candidates=4)
print("Candidate set:", candidates)
ranked = rank_candidates("Anna", candidates, rating_matrix, content_features, movies)
display(ranked)
final_list = rerank_for_diversity_and_freshness(ranked, movies, top_k=5)
final_list


## Part 7 - Short explanations and responsible recommendations

Responsible systems should make important recommendations understandable.

Example style from the lecture:

> "Recommended because you recently watched several science-fiction movies."

Below we generate a simple explanation from the user's liked genres and the chosen hybrid source.


In [ ]:
def explain_recommendation(user_id, title, rating_matrix, movies_df):
    meta = movies_df.set_index("title")
    user_ratings = rating_matrix.loc[user_id].dropna()
    liked = user_ratings[user_ratings >= 4]
    liked_titles = list(liked.index)
    liked_genres = movies_df.set_index("title").loc[liked_titles, "genre"] if len(liked_titles) else pd.Series(dtype=str)
    top_genre = liked_genres.mode().iloc[0] if not liked_genres.empty else None
    item_genre = meta.loc[title, "genre"]

    if top_genre is not None and item_genre == top_genre:
        reason = f"Recommended because you highly rated several {top_genre} movies."
    elif title in movies_df[movies_df["year"] >= 2023]["title"].values:
        reason = f"Recommended as a fresh {item_genre} title that matches your broader profile."
    else:
        reason = f"Recommended by combining collaborative patterns and {item_genre} content signals."

    return {
        "user": user_id,
        "title": title,
        "genre": item_genre,
        "liked_movies": ", ".join(liked_titles) if liked_titles else "(none)",
        "explanation": reason,
    }


top_title = final_list.iloc[0]["title"]
pd.DataFrame([explain_recommendation("Anna", top_title, rating_matrix, movies)])


## Part 8 - From prototype to production (concept map)

What we did in the notebook maps to production stages:

| Notebook piece | Production idea |
| --- | --- |
| ratings + movie metadata | data collection / feature store |
| CF + CB + popularity candidates | candidate generation |
| weighted score | ranking |
| diversity / freshness bonus | re-ranking |
| explanation text | transparency / user trust |
| Nora cold-start rule | fallback strategy |

A successful notebook model is only the beginning: production also needs latency control, monitoring (CTR, diversity, drift), and responsible deployment.


# Challenges

### Challenge 1 - Tune the weighted hybrid

**Goal:**
See how CF/CB weights change the Top recommendation.

**What to do:**

1. Run `weighted_hybrid("Anna", ..., w_cf=0.8, w_cb=0.2)`.
2. Then run `weighted_hybrid("Anna", ..., w_cf=0.2, w_cb=0.8)`.
3. Compare the Top-3 lists.
4. Decide which weight setting better fits a platform that wants more unexpected CF discoveries.


In [ ]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:



### Your observations

Which weight setting changed Anna's Top-3 most? When would you prefer a higher CB weight?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 - Improve the switching rule

**Goal:**
Design a clearer cold-start policy.

**What to do:**

1. Inspect Nora's number of ratings and current switching output.
2. Change the switching thresholds (for example, cold user if `n_ratings < 3`).
3. Add one extra rule: if the user liked only one genre, prefer content-based.
4. Compare recommendations for Nora and Anna after your change.


In [ ]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:



### Your observations

Did your new rule help Nora more than Anna? Why is switching useful for cold start?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 - Concept Check: Hybrid and Production

This challenge requires no programming.

Explain in your own words:

1. What is the difference between a **weighted** hybrid and a **mixed** hybrid?
2. Why do production systems use **candidate generation** before ranking?
3. Why are **explanations**, monitoring, and fallback strategies important after deployment?

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
